# BS5 frozen-policy continuous evaluation600
No training. Attach NEW source + full fresh60k audit + matching repro_full. Four cells.


In [ ]:
import os, sys, pathlib, json, subprocess, hashlib
os.environ['PYTHONDONTWRITEBYTECODE']='1'
INPUT=pathlib.Path('/kaggle/input'); WORK=pathlib.Path('/kaggle/working')
sources=sorted({p.parents[2] for p in INPUT.rglob('eval_bs5_continuous.py') if p.parent.name=='evaluation'})
assert len(sources)==1, f'Attach one NEW source dataset: {sources}'
SRC=sources[0]; os.chdir(SRC)
OUT=WORK/'bs5_continuous600'
def command(*args):
    subprocess.run([sys.executable,'-B','-m','scripts.evaluation.eval_bs5_continuous',*map(str,args)],cwd=SRC,check=True)
print('cd', SRC, 'output', OUT, 'EVALUATION ONLY; training0; horizon600')


In [ ]:
# Locate the full matching audit and repro; do not use historical same-seed models.
AUDIT_SHA='82797392e0a2febcc68f74bc64c4353c73ea540d8b191e47508058741ffa86e8'
audits=[]
for p in INPUT.rglob('protocol.json'):
    if not (p.parent/'runs/C-R_seed551/evaluations/057000/130001.json.gz').is_file(): continue
    data=json.loads(p.read_text())
    if data.get('continuation',{}).get('origin')=='fresh_0_to_60k_v1': audits.append(p.parent)
assert len(audits)==1, f'Attach one full fresh60k audit dataset: {audits}'
AUDIT=audits[0]
repros=[]
for p in INPUT.rglob('EVIDENCE_COMPANIONS.json'):
    if any(x.get('sha256')==AUDIT_SHA for x in json.loads(p.read_text())): repros.append(p.parent)
assert len(repros)<=1, f'Attach only one matching repro: {repros}'
REPRO=repros[0] if repros else AUDIT
import torch, numpy as np
print('Runtime:',sys.version,torch.__version__,np.__version__,'GPU:',torch.cuda.is_available())
if repros:
    assert torch.cuda.is_available(), 'Enable GPU before preparing a READY lineage.'
if not (OUT/'protocol.json').exists():
    command('prepare','--result-root',AUDIT,'--checkpoint-root',REPRO,'--output',OUT,'--device','auto')
selection=json.loads((OUT/'checkpoint_selection.json').read_text())
if selection['status']!='READY':
    print('BLOCKED. Missing weights:',json.dumps(selection['missing_weights'],indent=2))
    print('Attach repro_full_6d851d3f2b66.zip. Preserve this zero-step BLOCKED output; use a new OUT after adding weights.')
assert selection['status']=='READY', 'No control/main steps have run; no replacement checkpoint allowed.'
assert torch.cuda.is_available(), 'Enable GPU before preparing a READY main run.'
print('Locked4 best models. Next cell spends at most25200 eval steps, training0.')


In [ ]:
# Controls1200 + main24000. Re-run skips valid committed games only.
# A partial game is fail-closed: no automatic reset, replay, or budget extension.
command('run','--output',OUT,'--checkpoint-root',REPRO)


In [ ]:
# Final package is built once by cell3. This cell verifies/downloads it.
from IPython.display import FileLink, display
receipt=WORK/(OUT.name+'_export_receipt.json')
if receipt.is_file():
    info=json.loads(receipt.read_text()); package=pathlib.Path(info['path'])
    h=hashlib.sha256()
    with package.open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''): h.update(chunk)
    assert h.hexdigest()==info['sha256'] and info['accepted']
    display(FileLink(str(package))); display(FileLink(str(receipt)))
else:
    print('No complete600 bundle. Preserve output and inspect BLOCKED/INCOMPLETE evidence.')
for name in ['REPORT_VI.md','checkpoint_selection.json','budget_ledger.json','verification.json']:
    if (OUT/name).is_file(): display(FileLink(str(OUT/name)))
